# Mini-projeto 1 - Fase 2: CNN para classificação do CIFAR-10

Continuidade da Fase 1 (MLP, ver `../fase1-mlp/`). A lógica reutilizável (modelo, dados, treino, métricas, checkpointing) vive no pacote `cnn_cifar10` em `../src/`, seguindo exatamente o mesmo padrão da Fase 1 — o notebook fica focado em **definir experimentos e reportar resultados**, não em implementação.

**Integrantes do grupo:** _preencher aqui (nome de todos)_

O que este notebook cobre (conforme o enunciado do mini-projeto):
- Treino de uma CNN no CIFAR-10 com hiperparâmetros configuráveis (nº/tamanho de filtros, kernel size, stride, padding, pooling, dropout, taxa de aprendizagem, além dos já cobertos na Fase 1: ativação, otimizador, função de erro).
- Métricas por classe (acurácia) e globais (acurácia, precision, recall, f1).
- Comparação direta com o melhor resultado do MLP (Fase 1: ensemble `final` = 0.6135 de acurácia) — ver `../../fase1-mlp/README.md`.
- Cada execução de treino é salva automaticamente em `../results/` (pesos + config + métricas + histórico) — ver `../src/cnn_cifar10/checkpointing.py`.

**Recomendação forte: rode este notebook no Google Colab com GPU** (Ambiente de execução > Alterar tipo de ambiente de execução > GPU). CNN é bem mais lenta que MLP em CPU, e o ganho de GPU aqui é de 10-50x+.

## 0. Setup do ambiente

- **Local**: rode a partir de um ambiente onde o pacote já foi instalado (`pip install -e .` na pasta `fase2-cnn/`).
- **Google Colab**: a célula abaixo clona o repositório e instala o pacote automaticamente.

In [ ]:
#@title Setup (Colab ou local)
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/jpbezerra/redes-neurais.git"
    REPO_DIR = Path("/content/redes-neurais")
    if not REPO_DIR.exists():
        !git clone {REPO_URL} {REPO_DIR}
    else:
        !cd {REPO_DIR} && git pull
    %pip install -q -e {REPO_DIR}/miniprojeto/fase2-cnn
    PROJECT_ROOT = REPO_DIR / "miniprojeto" / "fase2-cnn"
else:
    PROJECT_ROOT = Path.cwd().parent  # notebooks/ -> fase2-cnn/
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
#@title Imports
import torch
import matplotlib.pyplot as plt
import pandas as pd

from cnn_cifar10.config import ExperimentConfig
from cnn_cifar10.data import get_dataloaders, CLASSES
from cnn_cifar10.train import fit, fit_or_load
from cnn_cifar10.checkpointing import load_all_metadata
from cnn_cifar10.utils import set_seed, get_device

In [ ]:
#@title Device
device = get_device()
print("Usando dispositivo:", device)
if device.type == "cpu":
    print("Aviso: sem GPU disponível. CNN em CPU é bem mais lenta — considere rodar no Colab.")

# No Colab/Linux, num_workers > 0 acelera o carregamento com augmentation
# (no Windows local, exige if __name__ == '__main__': em scripts, então mantemos 0 lá).
NUM_WORKERS = 2 if IN_COLAB else 0

## 1. Experimento baseline

Arquitetura equivalente à do notebook de referência do professor (`temp/CIFAR10_with_CNNs.ipynb`, adaptação do LeNet-5): 2 blocos convolucionais (32 e 64 filtros, kernel 3x3, padding 1, stride 1) + max pooling 2x2 após cada bloco, seguidos de cabeça densa `120 -> 84 -> 10`. ReLU, Adam, entropia cruzada, sem regularização — ponto de partida para a busca guiada, do mesmo jeito que a Fase 1 partiu do baseline MLP `[64,128,64]`.

In [ ]:
baseline_config = ExperimentConfig(
    run_name="baseline",
    conv_channels=(32, 64),
    kernel_size=3,
    stride=1,
    padding=1,
    pool_size=2,
    fc_layers=(120, 84),
    activation="relu",
    optimizer="adam",
    loss="cross_entropy",
    learning_rate=1e-3,
    batch_size=32,
    num_epochs=40,
    patience=5,
    notes="Baseline equivalente ao notebook de referencia (2 conv + 2 pool + 3 fc), ponto de partida da busca guiada da Fase 2.",
    tags=["baseline"],
)

set_seed(baseline_config.seed)
train_loader, val_loader, test_loader = get_dataloaders(
    data_dir=DATA_DIR,
    batch_size=baseline_config.batch_size,
    val_fraction=baseline_config.val_fraction,
    seed=baseline_config.seed,
    num_workers=NUM_WORKERS,
    augment=baseline_config.augment,
    normalization=baseline_config.normalization,
)

result_baseline = fit_or_load(
    baseline_config, train_loader, val_loader, test_loader, device,
    class_names=list(CLASSES), results_dir=RESULTS_DIR,
)
result_baseline["test_scores"]

## 2. Próximos experimentos

Seguir o mesmo padrão de busca guiada da Fase 1 (`scripts/run_experiments.py`): cada rodada informada pelo resultado da anterior, cobrindo os hiperparâmetros pedidos no enunciado — tamanho da rede (`conv_channels`/`fc_layers`), kernel size, stride, padding, dropout, pooling e taxa de aprendizagem. Cada experimento novo é só uma célula com um `ExperimentConfig` diferente + chamada a `fit_or_load` (idempotente: reexecutar a célula não retreina um `run_name` já salvo em `results/`).

Ideias de variações isoladas para começar (uma alavanca por vez, como na leva 1 do MLP):
- **Kernel size**: 5x5 em vez de 3x3.
- **Stride**: stride 2 no lugar de um pooling.
- **Padding**: sem padding (`padding=0`) vs. `same` (`padding=1` com kernel 3).
- **Pooling**: `pool_size=2` (padrão) vs. `pool_size=4` numa rede mais funda.
- **Profundidade**: 3 blocos convolucionais (ex.: `conv_channels=(32, 64, 128)`).
- **Dropout**: `dropout=0.2/0.3/0.5` (aplica em `Dropout2d` nos blocos conv e `Dropout` na cabeça densa).
- **Batch norm**: `batch_norm=True`.
- **Learning rate**: 5e-4 / 5e-3.
- **Data augmentation**: `augment=True` (mesma transform de crop+flip da Fase 1, ver `cnn_cifar10/data.py`).

Depois de mapear os hiperparâmetros isoladamente, combinar os vencedores em rodadas sucessivas (busca gulosa) como nas levas 2-8 do MLP — considerar mover para `scripts/run_experiments.py` se as rodadas ficarem muito lentas para caber numa célula (ver `../fase1-mlp/scripts/run_experiments.py` como referência de estrutura).

In [ ]:
#@title Comparar execuções salvas
df = load_all_metadata(RESULTS_DIR)
if not df.empty:
    cols = [c for c in ["run_name", "metrics.test_accuracy", "metrics.test_f1_score", "metrics.epochs_trained"] if c in df.columns]
    display(df[cols].sort_values("metrics.test_accuracy", ascending=False) if cols else df)
else:
    print("Nenhuma execucao salva ainda em", RESULTS_DIR)